In [2]:
using CMPSExcitations

In [3]:
Hsingle(c, μ) = ∫(2 * ∂ψ̂' * ∂ψ̂ - 2 * μ * ψ̂' * ψ̂ + 4 * c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));

In [276]:
c, μ = 10., 5.
tol = 1e-10
HLL = Hsingle(c, μ)

Ds = [8]
stateLL = find_groundstate(Ds, HLL, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
D = maximum(Ds)
println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])

Optimizing D=8


┌ Info: YangGaudinCMPS ground state: initialization with e = 608.368212375170
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 357 iterations: f = -2.761265509087, ‖∇f‖ = 8.0189e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 8 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
  4.459082 seconds (3.68 M allocations: 346.962 MiB, 1.11% gc time)
---------------
Energy density: -2.761265509087142
 Particle density: 0.43647721538078255
 Order parameter: -0.3617397658512992


┌ Info: YangGaudinCMPS ground state: converged after 358 iterations: e = -2.761265509087, ‖∇e‖ = 8.0189e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118


In [277]:
stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], stateLL.Rs[1]));
HCLL = Hcoupled(c, μ)
println("Energy density: ", expval(HCLL.h, stateCLL)[], "\nParticle density: ", expval(ψ̂₁' * ψ̂₁ + ψ̂₂' * ψ̂₂, stateCLL)[], "\nDensity imbalance: ", expval(ψ̂₁' * ψ̂₁ - ψ̂₂' * ψ̂₂, stateCLL)[])

Energy density: -2.7612655090871625
Particle density: 0.8729544307615631
Density imbalance: 0.0


In [278]:
# canonical basis
function projection_matrix(D, M)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E / sqrt(2) - M * Diagonal(F) / M
        W2 = E / sqrt(2) + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

function excitation_matrix_constrained(Heff, M)
    D = size(M, 1) # R = MDᵣ/M
    H = excitation_matrix(Heff, D)
    P = projection_matrix(D, M)

    P' * H * P, P
end

excitation_matrix_constrained (generic function with 1 method)

In [279]:
_, M = eigen(stateLL.Rs[1][]);
P = projection_matrix(D, M)

Q = P[1:D^2, D^2+1:end]
Qn = Matrix(qr(Q).Q)
Pn = copy(P)
Pn[1:D^2, D^2+1:end] .= Qn / sqrt(2)
Pn[D^2+1:end, D^2+1:end] .= -Qn / sqrt(2)

64×8 view(::Matrix{ComplexF64}, 65:128, 65:72) with eltype ComplexF64:
   0.0173304-2.47084e-17im  …     -0.23796-0.00668516im
 -0.00915711+3.43999e-18im      -0.0110328-0.0190576im
   0.0222215+5.39389e-18im       0.0271741-0.00952034im
  -0.0155876+2.0723e-18im       -0.0262117+0.0340141im
   0.0055077+8.35394e-18im     -0.00221717+0.0303086im
 -0.00915762-2.28724e-17im  …    -0.017663-0.0256483im
  0.00798631+1.11338e-17im        0.021335-0.00103199im
   0.0223409+3.51546e-17im       0.0497893-0.043004im
  -0.0974401-7.08927e-18im       0.0332353-0.00551895im
   0.0514858-1.37847e-18im       -0.232321-0.00789384im
            ⋮               ⋱  
    0.163425+1.70143e-16im  …   -0.0310224-0.0161176im
   0.0714934+3.15079e-18im      -0.0214066+0.00407057im
  -0.0377759+4.56256e-35im     -0.00472364+0.00186752im
   0.0916705+2.52063e-17im     -0.00316724+0.00489632im
  -0.0643037-3.15079e-18im       0.0102448-0.0135046im
    0.022721+9.45236e-18im  …    0.0159249+0.00738265im
   -0.037

In [280]:
space = InfiniteCMPSExcitationSpace(0, stateCLL, stateCLL)
Heff = excitation_matrix(excitation_operator(HCLL, space), D);

In [281]:
round.(Pn' * Heff * Pn, digits=3)

72×72 Matrix{ComplexF64}:
  75.336+0.0im     16.416+0.0im    …   0.457-1.306im   2.638+0.181im
  16.416+0.0im     40.776+0.0im       -0.392+0.316im  -2.327-0.02im
 -15.539+0.0im       5.68+0.0im        0.083+0.506im    0.54-0.092im
  -5.688+0.0im      -10.7+0.0im        0.104+0.234im   0.646-0.048im
 -11.768+0.0im    -12.642+0.0im        0.767+0.479im   4.648-0.144im
 -17.084+0.0im     -32.19+0.0im    …    0.85-1.275im   4.998+0.143im
   9.403+0.0im     13.757+0.0im        0.098+2.994im   0.839-0.51im
  19.249+0.0im    -14.694+0.0im        0.595+0.514im   3.615-0.136im
   -5.15+0.0im     -7.857+0.0im       -0.249+6.285im  -0.972-1.033im
   8.364+0.0im     -1.495+0.0im        0.357+2.348im    2.34-0.423im
        ⋮                          ⋱        ⋮         
 -30.957+0.0im      6.374+0.0im       -1.221+2.308im  -7.142-0.285im
  -0.495-0.0im     -1.439-0.0im       -0.933+0.545im  -5.561-0.013im
   0.417+0.0im      2.455-0.0im    …   0.944-9.616im   4.867+1.533im
   0.245+1.833im   -0.05

### Explicitly constructing goldstone modes

In [4]:
# canonical basis
function projection_matrix(D, M)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return Matrix(qr(P).Q) # moves to an orthogonal projection
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

function excitation_matrix_constrained(Heff, M)
    D = size(M, 1) # R = MDᵣ/M
    H = excitation_matrix(Heff, D)
    P = projection_matrix(D, M)

    P' * H * P, P
end

excitation_matrix_constrained (generic function with 1 method)

In [13]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
c, μ = 10., 5.
tol = 1e-10
Ds = [4, 8]
D = maximum(Ds)

HLL = Hsingle_ll(c, μ)
stateLL = find_groundstate(Ds, HLL, InfiniteCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])

HCLL = Hcoupled(c, μ)
stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], zero(stateLL.Rs[1])));
println("Energy density: ", expval(HCLL.h, stateCLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateCLL)[], "\n Order parameter: ", expval(ψ̂, stateCLL)[])

Optimizing D=4


┌ Info: UniformCMPS ground state: initialization with e = 8.144137352925
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:115
┌ Info: LBFGS: converged after 119 iterations: f = -2.734747817523, ‖∇f‖ = 9.2592e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
  0.181349 seconds (475.55 k allocations: 22.095 MiB, 9.34% gc time)
---------------
Optimizing D=8


┌ Info: UniformCMPS ground state: converged after 120 iterations: e = -2.734747817523, ‖∇e‖ = 9.2592e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127
┌ Info: UniformCMPS ground state: initialization with e = -2.734747817397
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:115
┌ Info: LBFGS: converged after 326 iterations: f = -2.761265509087, ‖∇f‖ = 9.8512e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 8 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
  5.978347 seconds (3.21 M allocations: 297.855 MiB, 0.92% gc time)
---------------
Energy density: -2.761265509087068
 Particle density: 0.8729544307594911
 Order parameter: -0.5115772829135019
Energy density: -2.761265509087068

┌ Info: UniformCMPS ground state: converged after 327 iterations: e = -2.761265509087, ‖∇e‖ = 9.8512e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127



 Particle density: 0.8729544307594911
 Order parameter: -0.5115772829135019


In [10]:
Q, R1, R2 = stateCLL.Q[], stateCLL.Rs[1][], stateCLL.Rs[2][];
W1, W2 = R1, -R2;

function leftgaugetv(Q, R, V, W1, W2, p)
    # assuming Q, R are in cMPS left gauge
    X, _ = linsolve(-R' * (W1 + W2) - V) do X
        (Q * X - X * Q) + 1im * p * X + 2 * R' * (R * X - X * R)
    end

    return V + (Q * X - X * Q) + 1im * p * X, W1 + (R * X - X * R), W2 + (R * X - X * R)
end

ps = range(0, 5., 20)
energies = zeros(ComplexF64, length(ps))
ρr = rightenv(stateCLL)[1][]

Threads.@threads for idx in eachindex(ps)
    V, W1, W2 = leftgaugetv(Q, R, zero(Q), R, -R, ps[idx])
    # ∫⟨H⟩ - ⟨H₀⟩ = E -> intensive contribution to energy on top of extensive ground state energy
    space = InfiniteCMPSExcitationSpace(ps[idx], stateCLL, stateCLL)
    Heff, P = excitation_matrix_constrained(excitation_operator(HCLL, space), eigen(stateCLL.Rs[1][]).vectors)

    # Heff maps between D^2 + D subspace; so P * Heff * P' maps between 2D^2 space while keeping the multispecies constraint satisfied
    Ws = P * Heff * P' * vcat(vec(W1), vec(W2))

    HW1 = reshape(view(Ws, 1:D^2), D, D)
    HW2 = reshape(view(Ws, (D^2+1):2*D^2), D, D)

    energies[idx] = tr(HW1 * ρr * W1' + HW2 * ρr * W2')
end

energies

20-element Vector{ComplexF64}:
  267.7644714157313 + 9.879887522402526e-15im
 267.99796810928564 - 0.27032069531812686im
  268.7242839800425 - 0.5818417937577746im
 269.99278613426185 - 0.9477441180471714im
 271.82804329409174 - 1.3737601206568355im
  274.2347953726598 - 1.8578743728722653im
 277.20840391915374 - 2.392581107789867im
  280.7407352476229 - 2.9686933991305686im
  284.8231119913563 - 3.5782391479849966im
 289.44793231142046 - 4.2160339067311865im
   294.609764980973 - 4.880069176219309im
  300.3061825215572 - 5.570912167008526im
   306.538362214568 - 6.290433726788483im
 313.31124770410844 - 7.040298638091088im
 320.63286690697964 - 7.820633929986598im
 328.51261621386146 - 8.62914935219753im
 336.95880213612696 - 9.460948821158285im
  345.9752993450914 - 10.309498701748163im
 355.55582600932064 - 11.168926875752215im
  365.6803065299287 - 12.03511110422965im

In [12]:
# plot(ps, real.(energies))